# Notebook 01 — Limpeza de Dados e Análise Exploratória
**Associação Passos Mágicos | PEDE 2022–2024**

Este notebook realiza a limpeza, padronização e exploração inicial dos dados coletados nas
Pesquisas Extensivas do Desenvolvimento Educacional (PEDE) dos anos de 2022, 2023 e 2024.
O produto final é um dataset unificado, salvo em `../data/dados_limpos.csv`, que alimentará
os demais notebooks de análise e modelagem.


## 1. Configuração do Ambiente

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.3f}".format)

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
})
PALETA = ["#1B4F72", "#2980B9", "#A9CCE3", "#F39C12", "#E74C3C", "#27AE60"]
sns.set_palette(PALETA)

DADOS_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
FIGURAS_DIR = os.path.join(os.path.dirname(os.getcwd()), "figuras")
os.makedirs(DADOS_DIR, exist_ok=True)
os.makedirs(FIGURAS_DIR, exist_ok=True)

ARQUIVO = os.path.join(os.path.dirname(os.getcwd()), "database.xlsx")
print(f"Arquivo localizado em: {ARQUIVO}")
print(f"Diretório de saída: {DADOS_DIR}")


## 2. Carregamento dos Dados Brutos

In [ ]:
# Carrega cada aba do arquivo Excel
df22_raw = pd.read_excel(ARQUIVO, sheet_name="PEDE2022")
df23_raw = pd.read_excel(ARQUIVO, sheet_name="PEDE2023")
df24_raw = pd.read_excel(ARQUIVO, sheet_name="PEDE2024")

print(f"PEDE 2022: {df22_raw.shape[0]:,} alunos | {df22_raw.shape[1]} variáveis")
print(f"PEDE 2023: {df23_raw.shape[0]:,} alunos | {df23_raw.shape[1]} variáveis")
print(f"PEDE 2024: {df24_raw.shape[0]:,} alunos | {df24_raw.shape[1]} variáveis")


In [ ]:
# Inspeciona as colunas de cada ano
for ano, df in [(2022, df22_raw), (2023, df23_raw), (2024, df24_raw)]:
    print(f"\n=== PEDE {ano} ===")
    print(list(df.columns))


## 3. Padronização das Colunas

In [ ]:
# Mapeamento para nomes canônicos (common schema)
MAP22 = {
    "RA": "ra", "Fase": "fase", "Turma": "turma", "Nome": "nome",
    "Ano nasc": "ano_nasc", "Idade 22": "idade", "Gênero": "genero",
    "Ano ingresso": "ano_ingresso", "Instituição de ensino": "instituicao",
    "Pedra 20": "pedra_20", "Pedra 21": "pedra_21", "Pedra 22": "pedra_ref",
    "INDE 22": "inde", "Cg": "cg", "Cf": "cf", "Ct": "ct",
    "Nº Av": "num_avaliadores", "Rec Av1": "rec_av1", "Rec Av2": "rec_av2",
    "Rec Av3": "rec_av3", "Rec Av4": "rec_av4",
    "IAA": "iaa", "IEG": "ieg", "IPS": "ips", "IDA": "ida",
    "Matem": "nota_mat", "Portug": "nota_port", "Inglês": "nota_ing",
    "Rec Psicologia": "rec_psico", "Indicado": "indicado_bolsa",
    "Atingiu PV": "atingiu_pv", "IPV": "ipv", "IAN": "ian",
    "Fase ideal": "fase_ideal", "Defas": "defasagem",
    "Destaque IEG": "destaque_ieg", "Destaque IDA": "destaque_ida",
    "Destaque IPV": "destaque_ipv",
}
MAP23 = {
    "RA": "ra", "Fase": "fase", "Turma": "turma",
    "Nome Anonimizado": "nome", "Data de Nasc": "data_nasc",
    "Idade": "idade", "Gênero": "genero", "Ano ingresso": "ano_ingresso",
    "Instituição de ensino": "instituicao",
    "Pedra 20": "pedra_20", "Pedra 21": "pedra_21",
    "Pedra 22": "pedra_22", "Pedra 23": "pedra_ref",
    "INDE 22": "inde_22", "INDE 23": "inde",
    "Cg": "cg", "Cf": "cf", "Ct": "ct",
    "Nº Av": "num_avaliadores", "Rec Av1": "rec_av1", "Rec Av2": "rec_av2",
    "Rec Av3": "rec_av3", "Rec Av4": "rec_av4",
    "IAA": "iaa", "IEG": "ieg", "IPS": "ips", "IPP": "ipp", "IDA": "ida",
    "Mat": "nota_mat", "Por": "nota_port", "Ing": "nota_ing",
    "Rec Psicologia": "rec_psico", "Indicado": "indicado_bolsa",
    "Atingiu PV": "atingiu_pv", "IPV": "ipv", "IAN": "ian",
    "Fase Ideal": "fase_ideal", "Defasagem": "defasagem",
    "Destaque IEG": "destaque_ieg", "Destaque IDA": "destaque_ida",
    "Destaque IPV": "destaque_ipv",
}
MAP24 = {
    "RA": "ra", "Fase": "fase", "Turma": "turma",
    "Nome Anonimizado": "nome", "Data de Nasc": "data_nasc",
    "Idade": "idade", "Gênero": "genero", "Ano ingresso": "ano_ingresso",
    "Instituição de ensino": "instituicao",
    "Pedra 20": "pedra_20", "Pedra 21": "pedra_21",
    "Pedra 22": "pedra_22", "Pedra 23": "pedra_23",
    "INDE 22": "inde_22", "INDE 23": "inde_23",
    "INDE 2024": "inde", "Pedra 2024": "pedra_ref",
    "Cg": "cg", "Cf": "cf", "Ct": "ct",
    "Nº Av": "num_avaliadores", "Rec Av1": "rec_av1",
    "IAA": "iaa", "IEG": "ieg", "IPS": "ips", "IPP": "ipp", "IDA": "ida",
    "Mat": "nota_mat", "Por": "nota_port", "Ing": "nota_ing",
    "Rec Psicologia": "rec_psico", "Indicado": "indicado_bolsa",
    "Atingiu PV": "atingiu_pv", "IPV": "ipv", "IAN": "ian",
    "Fase Ideal": "fase_ideal", "Defasagem": "defasagem",
    "Destaque IEG": "destaque_ieg", "Destaque IDA": "destaque_ida",
    "Destaque IPV": "destaque_ipv",
    "Escola": "escola", "Ativo/ Inativo": "status_ativo",
}

df22 = df22_raw.rename(columns=MAP22)
df23 = df23_raw.rename(columns=MAP23)
df24 = df24_raw.rename(columns=MAP24)

# Adiciona coluna de ano de referência
df22["ano"] = 2022
df23["ano"] = 2023
df24["ano"] = 2024

print("Colunas padronizadas com sucesso.")


## 4. Limpeza dos Dados

In [ ]:
# Colunas que devem estar em todos os datasets
COLUNAS_BASE = [
    "ra", "ano", "fase", "turma", "nome", "idade", "genero",
    "ano_ingresso", "instituicao", "pedra_ref", "inde",
    "iaa", "ieg", "ips", "ida", "ipv", "ian", "ipp",
    "nota_mat", "nota_port", "nota_ing",
    "indicado_bolsa", "atingiu_pv", "defasagem",
    "fase_ideal", "num_avaliadores",
]

def selecionar_colunas(df, cols):
    presentes = [c for c in cols if c in df.columns]
    ausentes = [c for c in cols if c not in df.columns]
    if ausentes:
        print(f"  Colunas ausentes: {ausentes}")
        for c in ausentes:
            df[c] = np.nan
    return df[cols].copy()

print("=== PEDE 2022 ===")
df22_base = selecionar_colunas(df22, COLUNAS_BASE)
print("=== PEDE 2023 ===")
df23_base = selecionar_colunas(df23, COLUNAS_BASE)
print("=== PEDE 2024 ===")
df24_base = selecionar_colunas(df24, COLUNAS_BASE)


In [ ]:
def limpar_dataframe(df, ano):
    d = df.copy()

    # Normaliza strings
    for col in ["genero", "instituicao", "pedra_ref", "indicado_bolsa",
                "atingiu_pv", "fase"]:
        if col in d.columns:
            d[col] = d[col].astype(str).str.strip().str.title()

    # Converte booleanos texto → 0/1
    for col in ["indicado_bolsa", "atingiu_pv"]:
        if col in d.columns:
            d[col] = d[col].map(
                lambda x: 1 if str(x).lower() in ["sim", "1", "true", "s", "yes"] else
                          0 if str(x).lower() in ["não", "nao", "0", "false", "n", "no"] else np.nan
            )

    # Padroniza gênero
    if "genero" in d.columns:
        d["genero"] = d["genero"].replace({
            "Menina": "Feminino", "Menino": "Masculino",
            "F": "Feminino", "M": "Masculino",
        })

    # Garante tipos numéricos nos indicadores
    indicadores = ["inde", "iaa", "ieg", "ips", "ida", "ipv", "ian", "ipp",
                   "nota_mat", "nota_port", "nota_ing", "defasagem"]
    for col in indicadores:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    # Normaliza IAN: era numérico (0–10), mantém como está
    # Cria variável target: risco de defasagem
    d["risco_defasagem"] = (
        (d["defasagem"].notna()) & (d["defasagem"] < 0)
    ).astype(int)

    # Anos na Passos Mágicos
    d["anos_na_pm"] = d["ano"] - d["ano_ingresso"].clip(upper=d["ano"])

    return d

df22_clean = limpar_dataframe(df22_base, 2022)
df23_clean = limpar_dataframe(df23_base, 2023)
df24_clean = limpar_dataframe(df24_base, 2024)

print("Limpeza concluída.")


## 5. Unificação dos Datasets

In [ ]:
df = pd.concat([df22_clean, df23_clean, df24_clean], ignore_index=True)
print(f"Dataset unificado: {df.shape[0]:,} registros | {df.shape[1]} variáveis")
print(f"\nDistribuição por ano:")
print(df["ano"].value_counts().sort_index())


In [ ]:
# Classifica pedra numericamente para análises ordinais
ORDEM_PEDRA = {"Quartzo": 1, "Ágata": 2, "Agata": 2, "Ametista": 3,
               "Topázio": 4, "Topazio": 4}
df["pedra_num"] = df["pedra_ref"].map(ORDEM_PEDRA)

# Normaliza texto da pedra
df["pedra_ref"] = df["pedra_ref"].replace({"Agata": "Ágata", "Topazio": "Topázio"})

# Classifica fase numericamente
def fase_para_num(fase):
    try:
        return int(str(fase).replace("Fase", "").strip())
    except:
        mapa = {"Alfa": 0, "Alpha": 0, "Universitário": 9,
                "Universitarios": 9, "Universitários": 9}
        return mapa.get(str(fase).strip(), np.nan)

df["fase_num"] = df["fase"].apply(fase_para_num)

print("Variáveis derivadas criadas.")
print(df.dtypes)


## 6. Diagnóstico de Valores Ausentes

In [ ]:
total = len(df)
nulos = df.isnull().sum()
pct_nulos = (nulos / total * 100).round(1)

diagnostico = pd.DataFrame({
    "Valores ausentes": nulos,
    "% ausente": pct_nulos
}).query("`Valores ausentes` > 0").sort_values("% ausente", ascending=False)

print(diagnostico.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cols_com_nulos = diagnostico[diagnostico["% ausente"] > 0].head(20)
ax.barh(cols_com_nulos.index[::-1], cols_com_nulos["% ausente"][::-1],
        color="#2980B9", edgecolor="white")
ax.set_xlabel("% de valores ausentes")
ax.set_title("Percentual de valores ausentes por variável", fontweight="bold")
for i, v in enumerate(cols_com_nulos["% ausente"][::-1]):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "01_valores_ausentes.png"), bbox_inches="tight")
plt.show()
print("Figura salva.")


## 7. Análise Exploratória dos Indicadores

In [ ]:
INDICADORES = ["inde", "iaa", "ieg", "ips", "ida", "ipv", "ian", "ipp"]
NOMES = {
    "inde": "INDE (Global)", "iaa": "IAA (Autoavaliação)",
    "ieg": "IEG (Engajamento)", "ips": "IPS (Psicossocial)",
    "ida": "IDA (Aprendizagem)", "ipv": "IPV (Ponto de Virada)",
    "ian": "IAN (Adequação ao Nível)", "ipp": "IPP (Psicopedagógico)",
}

# Estatísticas descritivas por ano
print("=== Estatísticas descritivas dos indicadores por ano ===\n")
for ind in INDICADORES:
    print(f"--- {NOMES[ind]} ---")
    tabela = df.groupby("ano")[ind].agg(["mean", "median", "std", "min", "max"])
    tabela.columns = ["Média", "Mediana", "Desvio padrão", "Mínimo", "Máximo"]
    print(tabela.round(3))
    print()


In [ ]:
# Distribuição dos indicadores — histogramas por ano
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
cores_anos = {2022: "#1B4F72", 2023: "#2980B9", 2024: "#A9CCE3"}

for i, ind in enumerate(INDICADORES):
    ax = axes[i]
    for ano, grp in df.groupby("ano"):
        grp[ind].dropna().plot.hist(
            ax=ax, bins=25, alpha=0.6, label=str(ano),
            color=cores_anos[ano], edgecolor="white"
        )
    ax.set_title(NOMES[ind], fontweight="bold", fontsize=10)
    ax.set_xlabel("Nota")
    ax.set_ylabel("Frequência")
    ax.legend(title="Ano", fontsize=8)

plt.suptitle("Distribuição dos Indicadores por Ano (2022–2024)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "02_distribuicao_indicadores.png"),
            bbox_inches="tight")
plt.show()


In [ ]:
# Matriz de correlação (indicadores)
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[INDICADORES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="Blues",
    linewidths=0.5, ax=ax, annot_kws={"size": 9},
    xticklabels=[n.split(" ")[0] for n in NOMES.values()],
    yticklabels=[n.split(" ")[0] for n in NOMES.values()],
)
ax.set_title("Matriz de Correlação dos Indicadores", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "03_correlacao_indicadores.png"),
            bbox_inches="tight")
plt.show()


In [ ]:
# Distribuição da classificação Pedra por ano
fig, ax = plt.subplots(figsize=(10, 5))
ordem_pedra = ["Quartzo", "Ágata", "Ametista", "Topázio"]
pedra_pct = (
    df.groupby(["ano", "pedra_ref"])
    .size()
    .reset_index(name="n")
    .assign(pct=lambda x: x.groupby("ano")["n"].transform(lambda s: s / s.sum() * 100))
)
pedra_pivot = pedra_pct.pivot(index="pedra_ref", columns="ano", values="pct").reindex(ordem_pedra)
pedra_pivot.plot(kind="bar", ax=ax, color=["#1B4F72", "#2980B9", "#A9CCE3"],
                 edgecolor="white", width=0.7)
ax.set_ylabel("% dos alunos")
ax.set_xlabel("")
ax.set_title("Distribuição por Classificação (Pedra) — 2022 a 2024",
             fontweight="bold")
ax.tick_params(axis="x", rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=8, padding=2)
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "04_distribuicao_pedra_ano.png"),
            bbox_inches="tight")
plt.show()


## 8. Salvando o Dataset Limpo

In [ ]:
saida = os.path.join(DADOS_DIR, "dados_limpos.csv")
df.to_csv(saida, index=False, encoding="utf-8-sig")
print(f"Dataset salvo em: {saida}")
print(f"Shape final: {df.shape}")
print(f"\nVariáveis disponíveis:")
for col in sorted(df.columns):
    nulos_pct = df[col].isnull().mean() * 100
    print(f"  {col:<25} dtype={str(df[col].dtype):<12} nulos={nulos_pct:.1f}%")
